# PV fault detection — Artificial Neural Network (ANN)

**Model settings:** Paper architecture (3 hidden layers: 74, 78, 91 neurons, ReLU; softmax output; Adam). Inputs are standardised. Learning rate 0.001. The number of epochs (max 1000) is chosen by early stopping on a validation slice of the training rows, then the network is retrained on all training rows for that many epochs.

**Data (all simulated):**
- *System 1 initial:* the paper's 4 features + Isc/G
- *System 1 final:* + I-V curve features (Vmp, Imp, FF) from the high-resolution sensor, + Isc/G and the three physics ratios
- *System 2:* trained on System 1, tested on System 2 (zero-shot and few-shot), 4 features + Isc/G

Every result is on rows the model never saw: 80 % train / 20 % test, 5 runs (like the paper).

In [1]:
# Where the data files are
import sys
if 'google.colab' in sys.modules:                 # running in Google Colab
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/PV_Project/PV_final'
else:                                             # running on your computer (VS Code / Jupyter)
    DATA_DIR = 'data'


## Load the data and build the inputs

In [2]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'   # hide TensorFlow start-up messages
import tensorflow as tf
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler

# Simulated datasets (made by pv_simulator_v2.py). Fault: 0 = normal, 1 = partial shading, 2 = dirt
s1_initial = pd.read_csv(f'{DATA_DIR}/S1_sim500_realistic_4features.csv')   # System 1: paper's 4 features
s1_final   = pd.read_csv(f'{DATA_DIR}/S1_sim500_realistic_hires_IV.csv')    # System 1: + I-V curve, high-res sensor
s2         = pd.read_csv(f'{DATA_DIR}/S2_sim200_realistic_4features.csv')   # System 2: paper's 4 features

def features(df):
    """Model inputs = the measured columns + the physics features they allow."""
    X = df.drop(columns='Fault').copy()
    X['Isc/G'] = X['Isc/MaxIsc'] / X['G/1000']                 # current per unit of sunlight
    if 'Vmp/MaxVoc' in X:                                       # I-V dataset: three more physics ratios
        X['Vmp/Voc'] = X['Vmp/MaxVoc'] / X['Voc/MaxVoc']        # shading moves the curve's knee
        X['Imp/Isc'] = X['Imp/MaxIsc'] / X['Isc/MaxIsc']        # shading: MPP current falls below Isc
        X['Imp/G']   = X['Imp/MaxIsc'] / X['G/1000']            # dirt: less MPP current per unit sunlight
    return X

LABELS = ['normal', 'shading', 'dirt']


## The model

In [3]:
def build(n_inputs, seed):
    tf.keras.utils.set_random_seed(seed)
    net = Sequential([Input((n_inputs,)), Dense(74, activation='relu'), Dense(78, activation='relu'),
                      Dense(91, activation='relu'), Dense(3, activation='softmax')])
    net.compile(Adam(0.001), 'categorical_crossentropy', metrics=['accuracy'])
    return net

class TrainedANN:
    def __init__(self, net, scaler): self.net, self.scaler = net, scaler
    def predict(self, X): return self.net.predict(self.scaler.transform(X), verbose=0).argmax(1)

def train_model(X, y, seed, weights=None):
    scaler = StandardScaler().fit(X)
    Xs, Y = scaler.transform(X), to_categorical(y, 3)
    w = np.ones(len(y)) if weights is None else np.asarray(weights)
    # 1) find a good number of epochs on a validation slice of the training rows
    a, b = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=100 + seed)
    net = build(X.shape[1], seed)
    hist = net.fit(Xs[a], Y[a], sample_weight=w[a], validation_data=(Xs[b], Y[b]), epochs=1000, batch_size=32, verbose=0,
                   callbacks=[EarlyStopping(monitor='val_loss', patience=100, restore_best_weights=True)])
    epochs = int(np.argmin(hist.history['val_loss'])) + 1
    # 2) retrain on all training rows for that many epochs
    net = build(X.shape[1], seed)
    net.fit(Xs, Y, sample_weight=w, epochs=epochs, batch_size=32, verbose=0)
    return TrainedANN(net, scaler)

## How each experiment is scored (5 runs)

In [4]:
def evaluate(data, title):
    """5 runs, each with a new 80/20 split (like the paper). Prints every run and the average."""
    X, y = features(data), data.Fault
    rows, matrices, reports = [], [], []
    for run in range(5):
        tr, te = train_test_split(X.index, test_size=0.2, stratify=y, random_state=run)
        model = train_model(X.loc[tr], y[tr], seed=run)
        pred = model.predict(X.loc[te])
        rows.append(dict(Run=run + 1, Accuracy=accuracy_score(y[te], pred),
                         Precision=precision_score(y[te], pred, average='weighted'),
                         Recall=recall_score(y[te], pred, average='weighted'),
                         F1=f1_score(y[te], pred, average='weighted')))
        matrices.append(confusion_matrix(y[te], pred))
        reports.append(classification_report(y[te], pred, target_names=LABELS, digits=4))
        print(f'Run {run + 1}: accuracy = {rows[-1]["Accuracy"]:.4f}')
    res = pd.DataFrame(rows).set_index('Run')
    print(f'\n===== {title} =====')
    print(res.round(4).to_string())
    print(f'\nMean of 5 runs:  ' + '   '.join(f'{m} = {res[m].mean():.4f}' for m in res.columns)
          + f'   (accuracy std = {res.Accuracy.std():.4f})')
    best = (res.Accuracy - res.Accuracy.mean()).abs().idxmin()          # the paper shows the run closest to the mean
    print(f'\nConfusion matrix, run {best} (closest to the mean). Rows = true class, columns = predicted:')
    print(pd.DataFrame(matrices[best - 1], index=['true ' + l for l in LABELS], columns=['pred ' + l for l in LABELS]).to_string())
    print(f'\nClassification report, run {best}:\n{reports[best - 1]}')
    return res.mean()


## System 1 — initial (4 features + Isc/G)

In [5]:
s1_init = evaluate(s1_initial, 'System 1 initial: 4 features + Isc/G')

Run 1: accuracy = 0.8000


Run 2: accuracy = 0.8200


Run 3: accuracy = 0.7900


Run 4: accuracy = 0.8200


Run 5: accuracy = 0.7400

===== System 1 initial: 4 features + Isc/G =====
     Accuracy  Precision  Recall      F1
Run                                     
1        0.80     0.7988    0.80  0.7962
2        0.82     0.8210    0.82  0.8173
3        0.79     0.7949    0.79  0.7876
4        0.82     0.8212    0.82  0.8165
5        0.74     0.7444    0.74  0.7311

Mean of 5 runs:  Accuracy = 0.7940   Precision = 0.7960   Recall = 0.7940   F1 = 0.7897   (accuracy std = 0.0329)

Confusion matrix, run 3 (closest to the mean). Rows = true class, columns = predicted:
              pred normal  pred shading  pred dirt
true normal            31             0          3
true shading            4            22          7
true dirt               3             4         26

Classification report, run 3:
              precision    recall  f1-score   support

      normal     0.8158    0.9118    0.8611        34
     shading     0.8462    0.6667    0.7458        33
        dirt     0.7222    0.7879    

## System 1 — final (I-V + high-res sensor + physics features)

In [6]:
s1_fin = evaluate(s1_final, 'System 1 final: I-V + hi-res sensor + Isc/G + physics ratios')

Run 1: accuracy = 0.9800


Run 2: accuracy = 0.9500


Run 3: accuracy = 0.9600


Run 4: accuracy = 1.0000


Run 5: accuracy = 0.9400

===== System 1 final: I-V + hi-res sensor + Isc/G + physics ratios =====
     Accuracy  Precision  Recall      F1
Run                                     
1        0.98     0.9811    0.98  0.9800
2        0.95     0.9503    0.95  0.9500
3        0.96     0.9610    0.96  0.9599
4        1.00     1.0000    1.00  1.0000
5        0.94     0.9439    0.94  0.9397

Mean of 5 runs:  Accuracy = 0.9660   Precision = 0.9673   Recall = 0.9660   F1 = 0.9659   (accuracy std = 0.0241)

Confusion matrix, run 3 (closest to the mean). Rows = true class, columns = predicted:
              pred normal  pred shading  pred dirt
true normal            33             0          1
true shading            0            33          0
true dirt               3             0         30

Classification report, run 3:
              precision    recall  f1-score   support

      normal     0.9167    0.9706    0.9429        34
     shading     1.0000    1.0000    1.0000        33
        dirt 

## System 2 — zero-shot and few-shot
Zero-shot: trained on System 1 only. Few-shot k: k System 2 rows per class added to training.

In [7]:
X1, y1 = features(s1_initial), s1_initial.Fault
X2, y2 = features(s2), s2.Fault

# Zero-shot: trained on System 1 only, tested on all 200 System 2 rows (5 runs)
zero = []
for run in range(5):
    model = train_model(X1, y1, seed=run)
    pred = model.predict(X2)
    zero.append(accuracy_score(y2, pred))
    print(f'Zero-shot run {run + 1}: accuracy = {zero[-1]:.4f}')
print(f'Zero-shot mean = {np.mean(zero):.4f}')
print('Confusion matrix of the last run (rows = true, columns = predicted):')
print(pd.DataFrame(confusion_matrix(y2, pred), index=['true ' + l for l in LABELS], columns=['pred ' + l for l in LABELS]).to_string())

# Few-shot: add k System 2 rows per class to training (weighted 5x), test on the other System 2 rows (5 runs)
few = {}
for k in [5, 10]:
    accs = []
    for run in range(5):
        shots = s2.groupby('Fault', group_keys=False).sample(k, random_state=100 * k + run)
        rest = s2.drop(shots.index)
        X = pd.concat([X1, features(shots)], ignore_index=True)
        y = pd.concat([y1, shots.Fault], ignore_index=True)
        w = np.r_[np.ones(len(X1)), np.full(len(shots), 5.0)]
        model = train_model(X, y, seed=run, weights=w)
        accs.append(accuracy_score(rest.Fault, model.predict(features(rest))))
    few[k] = np.mean(accs)
    print(f'\nFew-shot k={k}: ' + ', '.join(f'{a:.4f}' for a in accs) + f'   mean = {few[k]:.4f}')


Zero-shot run 1: accuracy = 0.7200


Zero-shot run 2: accuracy = 0.7600


Zero-shot run 3: accuracy = 0.7450


Zero-shot run 4: accuracy = 0.6900


Zero-shot run 5: accuracy = 0.7850
Zero-shot mean = 0.7400
Confusion matrix of the last run (rows = true, columns = predicted):
              pred normal  pred shading  pred dirt
true normal            66             0          1
true shading           35            28          4
true dirt               3             0         63



Few-shot k=5: 0.9838, 0.9946, 0.9622, 0.9351, 0.9892   mean = 0.9730



Few-shot k=10: 0.9765, 0.9941, 0.9941, 0.9882, 1.0000   mean = 0.9906


## Summary

In [8]:
summary = pd.DataFrame({
    'System 1 initial (4 features + Isc/G)': [s1_init['Accuracy']],
    'System 1 final (I-V + hi-res + physics)': [s1_fin['Accuracy']],
    'System 2 zero-shot': [np.mean(zero)],
    'System 2 few-shot k=5': [few[5]],
    'System 2 few-shot k=10': [few[10]],
}, index=['ANN']).round(4)
print(summary.T.to_string())
print('\nPaper (real data): System 1 accuracy 0.9833 (precision 0.9843) | System 2 zero-shot 0.5267')


                                            ANN
System 1 initial (4 features + Isc/G)    0.7940
System 1 final (I-V + hi-res + physics)  0.9660
System 2 zero-shot                       0.7400
System 2 few-shot k=5                    0.9730
System 2 few-shot k=10                   0.9906

Paper (real data): System 1 accuracy 0.9833 (precision 0.9843) | System 2 zero-shot 0.5267
